In [ ]:
import boto3
import csv
import io
import difflib
import re
import json
from datetime import datetime

s3 = boto3.client('s3')

# ==============================
# SECCIÓN 1: CONFIGURACIÓN Y DATOS
# ==============================

UBICACIONES = {
    "Palmar de Bravo": "https://maps.app.goo.gl/7v6Ef4fjUXfikHoc6",
    "Palmarito": "https://maps.app.goo.gl/4u6u3dygTy9ix1Jc9",
    "Tuzuapan": "https://maps.app.goo.gl/nRDEFtrzDSVk2rjx6",
    "Cuacnopalan": "https://maps.app.goo.gl/FdEd5rqw5sdQxpiB6",
    "San Jose Ixtapa": "Calle 8 norte no.107 San Jose Ixtapa",
    "Xaltepec": "https://maps.app.goo.gl/XbXaHgwrfCfcja2w6",
}

CONTACTO_EXTRA = "Número de contacto para Tecamachalco: 2494226136"

SUCURSALES_VALIDAS = list(UBICACIONES.keys())

# ==============================
# SECCIÓN 2: UTILIDADES DE SLOTS
# ==============================

def get_slot_value(slots, slot_name):
    """Extrae el valor de un slot de forma segura"""
    slot = slots.get(slot_name)
    if slot and slot.get('value'):
        return slot['value'].get('interpretedValue') or slot['value'].get('originalValue')
    return None

def build_slot(value):
    """Construye un objeto slot con valor"""
    return {
        "value": {
            "interpretedValue": value,
            "originalValue": value
        }
    }

def elicit_slot(intent_name, slots, slot_to_elicit, message):
    """Respuesta para pedir un slot específico"""
    return {
        "sessionState": {
            "dialogAction": {
                "type": "ElicitSlot",
                "slotToElicit": slot_to_elicit
            },
            "intent": {
                "name": intent_name,
                "state": "InProgress",
                "slots": slots
            }
        },
        "messages": [
            {
                "contentType": "PlainText",
                "content": message
            }
        ]
    }

def close_intent(intent_name, slots, message):
    """Cierra el intent con un mensaje final"""
    return {
        "sessionState": {
            "dialogAction": {
                "type": "Close"
            },
            "intent": {
                "name": intent_name,
                "state": "Fulfilled",
                "slots": slots
            }
        },
        "messages": [
            {
                "contentType": "PlainText",
                "content": message
            }
        ]
    }

# ==============================
# SECCIÓN 3: BÚSQUEDA DE ESTUDIOS EN CSV
# ==============================

def extract_estudio_name(text):
    """Extrae el nombre del estudio ignorando palabras de pregunta"""
    if not text:
        return ""

    stop_words = {
        'cual', 'es', 'el', 'precio', 'cuanto', 'cuesta',
        'valor', 'costó', 'qué', 'quiero', 'saber', 'dame', 'necesito',
        'por', 'favor', '?', '¿', 'podrías', 'decirme', 'informacion',
        'información', 'cotizacion', 'cotización', 'me', 'puedes', 'dar',
        'buenas', 'tardes', 'noches', 'días', 'hola', 'saludos', 'gracias',
        'agendar', 'cita', 'quiero', 'una', 'para', 'un'
    }

    patterns = [
        r'(?:precio|cuanto|cuesta|valor|precio de|precio del|precio de la)\s+(.+)',
        r'(?:cual es el precio de|necesito el precio de|quiero saber el precio de)\s+(.+)',
        r'(?:cuanto (?:cuesta|vale) (?:el|la)?)\s*(.+)',
        r'(?:agendar|cita para|cita de)\s+(.+)',
        r'(?:necesito|quiero)\s+(?:una\s+)?cita\s+(?:de|para)\s+(.+)',
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            resultado = match.group(1).strip()
            resultado = re.sub(r'[?¿]+$', '', resultado).strip()
            resultado = re.sub(r'^\s*de\s+', '', resultado)
            if resultado:
                return resultado

    words = text.split()
    filtered_words = [
        w for w in words
        if w.lower() not in stop_words and not re.match(r'^[?¿!.,]+$', w)
    ]
    resultado = ' '.join(filtered_words).strip()
    resultado = re.sub(r'\s+', ' ', resultado)
    resultado = re.sub(r'^\s*(?:de|del|la|el|y|por)\s+', '', resultado)
    return resultado

def buscar_estudio_en_csv(estudio_usuario, bucket_name, file_key):
    """Busca un estudio médico en cualquier CSV"""
    try:
        response = s3.get_object(Bucket=bucket_name, Key=file_key)
        contenido = response['Body'].read().decode('utf-8')
        reader = csv.DictReader(io.StringIO(contenido))
        estudios = [fila for fila in reader]

        if not estudios:
            return None

        # Detectar columna de nombre
        nombre_columna = None
        for col in estudios[0].keys():
            if 'estudio' in col.lower() or 'nombre' in col.lower():
                nombre_columna = col
                break
        if not nombre_columna:
            nombre_columna = list(estudios[0].keys())[0]

        estudios_dict = {}
        nombres_estudios = []

        for fila in estudios:
            nombre_original = fila[nombre_columna].strip()
            precio = fila.get('PRECIO', fila.get('precio', 'N/A'))
            condiciones = fila.get('CONDICIONES', fila.get('condiciones', 'No especificadas'))
            nombre_normalizado = nombre_original.lower()

            info = {'estudio': nombre_original, 'precio': precio, 'condiciones': condiciones}
            estudios_dict[nombre_normalizado] = info
            nombres_estudios.append(nombre_normalizado)

            # Variaciones adicionales
            variaciones = [
                re.sub(r'\b(de|del|la|el|y|por)\b', ' ', nombre_normalizado),
                re.sub(r'[^\w\s]', '', nombre_normalizado)
            ]
            for v in variaciones:
                v = re.sub(r'\s+', ' ', v).strip()
                if v and v not in estudios_dict and len(v) > 2:
                    estudios_dict[v] = info
                    nombres_estudios.append(v)

        estudio_normalizado = re.sub(r'\s+', ' ', estudio_usuario.lower().strip())
        resultado = None

        # 1. Exacta
        if estudio_normalizado in estudios_dict:
            resultado = estudios_dict[estudio_normalizado]

        # 2. Contenido
        if not resultado:
            for nombre in nombres_estudios:
                if (estudio_normalizado in nombre or nombre in estudio_normalizado) and len(nombre) > 3:
                    resultado = estudios_dict[nombre]
                    break

        # 3. Difusa
        if not resultado:
            coincidencias = difflib.get_close_matches(estudio_normalizado, nombres_estudios, n=3, cutoff=0.6)
            if coincidencias:
                resultado = estudios_dict[coincidencias[0]]

        # 4. Palabras clave
        if not resultado:
            palabras_usuario = {p for p in estudio_normalizado.split() if len(p) > 2}
            mejor_puntaje, mejor_estudio = 0, None
            for nombre in nombres_estudios:
                palabras_estudio = {p for p in nombre.split() if len(p) > 2}
                puntaje = len(palabras_usuario & palabras_estudio)
                if puntaje > mejor_puntaje:
                    mejor_puntaje = puntaje
                    mejor_estudio = nombre
            if mejor_estudio and mejor_puntaje >= 1:
                resultado = estudios_dict[mejor_estudio]

        return resultado

    except Exception as e:
        print(f"Error buscando estudio en {file_key}: {e}")
        return None

def guardar_cita_en_s3(cita_data, bucket_name):
    """Guarda la cita en formato JSON en S3"""
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        file_key = f"citas/cita_{timestamp}.json"
        cita_json = json.dumps(cita_data, ensure_ascii=False, indent=2)
        s3.put_object(
            Bucket=bucket_name,
            Key=file_key,
            Body=cita_json.encode('utf-8'),
            ContentType='application/json'
        )
        print(f"Cita guardada en: {file_key}")
        return True
    except Exception as e:
        print(f"Error guardando cita: {e}")
        return False

# ==============================
# SECCIÓN 4: FLUJO DE CITAS (CON RECOLECCIÓN DE DATOS)
# ==============================

def manejar_consulta_cita(intent_name, input_transcript, slots, bucket_name):
    """
    Flujo de citas con slots en orden:
      1. estudio   → buscar en CSV
      2. nombre    → nombre del paciente
      3. telefono  → teléfono de contacto
      4. sucursal  → sucursal deseada
      5. Guardar y confirmar
    """

    # --- PASO 1: Slot "estudio" ---
    print(f"[CITAS] Slots recibidos: {json.dumps(slots, ensure_ascii=False)}")
    print(f"[CITAS] Transcript: '{input_transcript}'")

    estudio_valor = get_slot_value(slots, 'estudio')
    print(f"[CITAS] estudio_valor desde slot: '{estudio_valor}'")

    if not estudio_valor:
        palabras_invalidas = {'estudio', 'cita', 'agendar', 'quiero', 'una', 'necesito', ''}

        # Primero intentar extraer con patrones (ej: "quiero cita de ultrasonido")
        estudio_extraido = extract_estudio_name(input_transcript)
        print(f"[CITAS] estudio_extraido del transcript: '{estudio_extraido}'")

        if estudio_extraido and estudio_extraido.lower().strip() not in palabras_invalidas and len(estudio_extraido) > 2:
            estudio_valor = estudio_extraido
        else:
            # El usuario respondió directo con el nombre (ej: "ultrasonido")
            transcript_limpio = re.sub(r'[¿?¡!.,]+', '', input_transcript).strip()
            if transcript_limpio and transcript_limpio.lower() not in palabras_invalidas and len(transcript_limpio) > 2:
                estudio_valor = transcript_limpio
                print(f"[CITAS] Usando transcript directo como estudio: '{estudio_valor}'")
            else:
                return elicit_slot(
                    intent_name, slots, 'estudio',
                    "¿Qué estudio médico deseas agendar? Por ejemplo: ultrasonido, rayos X, tomografía."
                )

        slots['estudio'] = build_slot(estudio_valor)

    # Verificar que el estudio exista en la base de citas
    resultado_estudio = buscar_estudio_en_csv(estudio_valor, bucket_name, "estudiocitas.csv")

    if not resultado_estudio:
        # Obtener sugerencias
        try:
            response = s3.get_object(Bucket=bucket_name, Key="estudiocitas.csv")
            contenido = response['Body'].read().decode('utf-8')
            reader = csv.DictReader(io.StringIO(contenido))
            estudios = [fila for fila in reader]
            nombre_columna = None
            for col in estudios[0].keys():
                if 'estudio' in col.lower() or 'nombre' in col.lower():
                    nombre_columna = col
                    break
            if not nombre_columna:
                nombre_columna = list(estudios[0].keys())[0]
            palabras = set(estudio_valor.lower().split())
            relevantes = [
                e[nombre_columna] for e in estudios
                if any(p in e[nombre_columna].lower() for p in palabras if len(p) > 3)
            ]
            sugerencias = ", ".join(relevantes[:5]) if relevantes else ", ".join(
                [estudios[i][nombre_columna] for i in range(min(5, len(estudios)))]
            )
        except Exception:
            sugerencias = "ultrasonido, rayos X, tomografía"

        # Limpiar slot para volver a preguntar
        slots['estudio'] = None
        return elicit_slot(
            intent_name, slots, 'estudio',
            f"No encontré '{estudio_valor}' en los estudios que requieren cita. "
            f"Estudios disponibles: {sugerencias}. ¿Cuál te interesa?"
        )

    # Confirmar estudio válido en el slot
    slots['estudiocita'] = build_slot(resultado_estudio['estudio'])

    # --- PASO 2: Slot "nombre" ---
    nombre_valor = get_slot_value(slots, 'nombre')
    if not nombre_valor:
        return elicit_slot(
            intent_name, slots, 'nombre',
            f"Perfecto, agendaremos un {resultado_estudio['estudio']} "
            f"(${resultado_estudio['precio']}). ¿Cuál es tu nombre completo?"
        )

    # --- PASO 3: Slot "telefono" ---
    telefono_valor = get_slot_value(slots, 'telefono')
    if not telefono_valor:
        return elicit_slot(
            intent_name, slots, 'telefono',
            f"Gracias, {nombre_valor}. ¿Cuál es tu número de teléfono de contacto?"
        )

    # --- PASO 4: Slot "sucursal" ---
    sucursal_valor = get_slot_value(slots, 'sucursal')
    if not sucursal_valor:
        opciones = ", ".join(SUCURSALES_VALIDAS)
        return elicit_slot(
            intent_name, slots, 'sucursal',
            f"¿En qué sucursal prefieres tu cita?\nOpciones: {opciones}"
        )

    # Validar que la sucursal sea válida (búsqueda flexible)
    sucursal_normalizada = sucursal_valor.lower().strip()
    sucursal_encontrada = None
    for s in SUCURSALES_VALIDAS:
        if sucursal_normalizada in s.lower() or s.lower() in sucursal_normalizada:
            sucursal_encontrada = s
            break
    if not sucursal_encontrada:
        # Búsqueda difusa
        coincidencias = difflib.get_close_matches(sucursal_normalizada, [s.lower() for s in SUCURSALES_VALIDAS], n=1, cutoff=0.5)
        if coincidencias:
            sucursal_encontrada = next(s for s in SUCURSALES_VALIDAS if s.lower() == coincidencias[0])
        else:
            opciones = ", ".join(SUCURSALES_VALIDAS)
            slots['sucursal'] = None
            return elicit_slot(
                intent_name, slots, 'sucursal',
                f"No reconocí esa sucursal. Por favor elige una de estas: {opciones}"
            )

    slots['sucursal'] = build_slot(sucursal_encontrada)

    # --- PASO 5: Guardar cita completa ---
    cita_data = {
        "estudio": resultado_estudio['estudio'],
        "precio": resultado_estudio['precio'],
        "condiciones": resultado_estudio['condiciones'],
        "paciente": nombre_valor,
        "telefono": telefono_valor,
        "sucursal": sucursal_encontrada,
        "fecha_solicitud": datetime.now().isoformat(),
        "estado": "pendiente",
        "fecha_cita": "por_confirmar"
    }

    exito = guardar_cita_en_s3(cita_data, bucket_name)

    if exito:
        mensaje = (
            f"✅ ¡Cita registrada exitosamente!\n\n"
            f"📋 Estudio: {resultado_estudio['estudio']}\n"
            f"💰 Precio: ${resultado_estudio['precio']}\n"
            f"📝 Condiciones: {resultado_estudio['condiciones']}\n"
            f"👤 Paciente: {nombre_valor}\n"
            f"📞 Teléfono: {telefono_valor}\n"
            f"📍 Sucursal: {sucursal_encontrada}\n\n"
            f"Te contactaremos pronto para confirmar fecha y hora."
        )
    else:
        mensaje = (
            f"Ocurrió un error al guardar tu cita. "
            f"Por favor llama directamente a tu sucursal en {sucursal_encontrada}."
        )

    return close_intent(intent_name, slots, mensaje)

# ==============================
# SECCIÓN 5: CONSULTA DE PRECIOS (estudios regulares)
# ==============================

def manejar_consulta_estudio(intent_name, input_transcript, bucket_name):
    """Maneja la consulta de precios de estudios médicos regulares"""

    estudio_usuario = extract_estudio_name(input_transcript)

    if not estudio_usuario:
        return elicit_slot(
            intent_name,
            {"estudio": None},
            'estudio',
            "¿Qué estudio médico te interesa consultar? Por ejemplo: examen general de orina, biometría hemática."
        )

    resultado = buscar_estudio_en_csv(estudio_usuario, bucket_name, "estudios (1).csv")

    if resultado:
        mensaje = (
            f"💵 {resultado['estudio']}\n"
            f"💰 Precio: ${resultado['precio']}\n"
            f"📋 Condiciones: {resultado['condiciones']}"
        )
        return close_intent(
            intent_name,
            {"estudio": build_slot(resultado['estudio'])},
            mensaje
        )
    else:
        # Sugerencias
        try:
            response = s3.get_object(Bucket=bucket_name, Key="estudios (1).csv")
            contenido = response['Body'].read().decode('utf-8')
            reader = csv.DictReader(io.StringIO(contenido))
            estudios = [fila for fila in reader]
            palabras = set(estudio_usuario.lower().split())
            relevantes = [
                e['ESTUDIO'] for e in estudios
                if any(p in e['ESTUDIO'].lower() for p in palabras if len(p) > 3)
            ]
            sugerencias = ", ".join(relevantes[:5]) if relevantes else ", ".join(
                [estudios[i]['ESTUDIO'] for i in range(min(5, len(estudios)))]
            )
        except Exception:
            sugerencias = "examen general de orina, biometría hemática, química sanguínea"

        return elicit_slot(
            intent_name,
            {"estudio": None},
            'estudio',
            f"No encontré información para '{estudio_usuario}'. Te sugiero: {sugerencias}. ¿Cuál te interesa?"
        )

# ==============================
# SECCIÓN 6: UBICACIONES / SUCURSALES
# ==============================

def responder_ubicacion(texto_usuario):
    """Responde con enlaces de Google Maps para ubicaciones"""

    texto_lower = texto_usuario.lower()

    if 'palmar de bravo' in texto_lower or ('palmar' in texto_lower and 'palmarito' not in texto_lower):
        mensaje = f"📍 Sucursal Palmar de Bravo\n🗺️ {UBICACIONES['Palmar de Bravo']}"
    elif 'palmarito' in texto_lower:
        mensaje = f"📍 Sucursal Palmarito\n🗺️ {UBICACIONES['Palmarito']}"
    elif 'tuzuapan' in texto_lower:
        mensaje = f"📍 Sucursal Tuzuapan\n🗺️ {UBICACIONES['Tuzuapan']}"
    elif 'cuacnopalan' in texto_lower:
        mensaje = f"📍 Sucursal Cuacnopalan\n🗺️ {UBICACIONES['Cuacnopalan']}"
    elif 'san jose ixtapa' in texto_lower or 'san josé ixtapa' in texto_lower:
        mensaje = f"📍 Sucursal San Jose Ixtapa\n📌 {UBICACIONES['San Jose Ixtapa']}"
    elif 'xaltepec' in texto_lower:
        mensaje = f"📍 Sucursal Xaltepec\n🗺️ {UBICACIONES['Xaltepec']}"
    elif 'tecamachalco' in texto_lower:
        mensaje = f"{CONTACTO_EXTRA}\n\n(Tecamachalco es atendido por otra empresa)"
    else:
        lineas = "\n".join([f"• {nombre}: {link}" for nombre, link in UBICACIONES.items()])
        mensaje = f"📍 Nuestras sucursales:\n\n{lineas}\n\n¿Cuál te interesa?"

    return {
        "sessionState": {
            "dialogAction": {"type": "Close"},
            "intent": {
                "name": "Dudas_generales",
                "state": "Fulfilled",
                "slots": {}
            }
        },
        "messages": [{"contentType": "PlainText", "content": mensaje}]
    }

# ==============================
# SECCIÓN 7: HANDLER PRINCIPAL
# ==============================

def lambda_handler(event, context):
    bucket_name = 'preciosdeestudios'

    print("Evento completo recibido:")
    print(json.dumps(event, ensure_ascii=False))

    try:
        input_transcript = event.get('inputTranscript', '')
        intent_name = event['sessionState']['intent']['name']
        # Leer slots actuales (pueden ser None si Lex aún no los ha rellenado)
        slots = event['sessionState']['intent'].get('slots') or {}
        # Normalizar: convertir valores None del JSON a None de Python
        slots = {k: v for k, v in slots.items()}

        print(f"Intent: {intent_name}")
        print(f"Transcript: '{input_transcript}'")
        print(f"Slots recibidos: {json.dumps(slots, ensure_ascii=False)}")

        # 1. Intent de citas → flujo con recolección de datos
        if intent_name == "Citas":
            return manejar_consulta_cita(intent_name, input_transcript, slots, bucket_name)

        # 2. Intent de dudas generales → ubicaciones
        if intent_name == "Dudas_generales":
            return responder_ubicacion(input_transcript)

        # 3. Cualquier intent con palabras de ubicación
        palabras_ubicacion = [
            'direccion', 'ubicacion', 'donde', 'mapa', 'google maps',
            'sucursal', 'dirección', 'ubicación', 'location',
            'palmar', 'tuzuapan', 'xaltepec', 'cuacnopalan', 'ixtapa', 'palmarito'
        ]
        if any(p in input_transcript.lower() for p in palabras_ubicacion):
            return responder_ubicacion(input_transcript)

        # 4. Por defecto → consulta de precios
        return manejar_consulta_estudio(intent_name, input_transcript, bucket_name)

    except Exception as e:
        import traceback
        print(f"Error: {e}")
        print(traceback.format_exc())

        intent_name_fallback = event.get('sessionState', {}).get('intent', {}).get('name', 'Consulta_estudio')
        return {
            "sessionState": {
                "dialogAction": {"type": "ElicitSlot", "slotToElicit": "estudio"},
                "intent": {
                    "name": intent_name_fallback,
                    "state": "InProgress",
                    "slots": {"estudio": None}
                }
            },
            "messages": [
                {
                    "contentType": "PlainText",
                    "content": "Ocurrió un error. Por favor dime nuevamente qué estudio necesitas."
                }
            ]
        }